# Приложение Б. Скрипт количественного анализа публикаций ВКонтакте

## 1. Установка зависимостей и импорт библиотек

In [ ]:
!pip install openpyxl statsmodels scikit-learn --quiet

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
from sklearn.metrics import roc_auc_score
import os

os.makedirs('output', exist_ok=True)


## 2. Загрузка и подготовка данных

Входной файл — `posts_coded.xlsx`, лист `Coded_Posts`.  
Зависимая переменная: `period_bin` (1 — период полярной ночи, 0 — контрольный период).

In [ ]:
df = pd.read_excel("posts_coded.xlsx", sheet_name="Coded_Posts")
df = df[df["period"].isin(["experimental", "control"])].copy()
df["period_bin"] = (df["period"] == "experimental").astype(int)

print(f"Всего постов: {len(df)}")
print(df["period"].value_counts())
df.head()


## 3. Метрики вовлечённости по периодам

Среднее, медиана, стандартное отклонение, квартили Q25 и Q75, минимум и максимум по лайкам, репостам, комментариям и просмотрам.

In [ ]:
metrics = ["likes", "reposts", "comments", "views"]
engagement = df.groupby("period")[metrics].agg([
    "count", "mean", "median", "std", "min", "max",
    lambda x: x.quantile(0.25),
    lambda x: x.quantile(0.75)
])
display(engagement)


## 4. Словарь блоков кодирования

In [ ]:
ALL_BLOCKS = {
    "Ритмика и сенсорика": [
        "SYNC", "TIME_WEEKEND", "TIME_EVENING",
        "CONTEXT_WEATHER", "CONTEXT_LIGHT", "CONTEXT_SEASON",
        "SENSORY_BODY", "SENSORY_WARM", "SENSORY_LIGHT"
    ],
    "Функции": ["FUNC_PRM", "FUNC_ENG", "FUNC_IMG", "FUNC_COM", "FUNC_INF"],
    "Фокус": [
        "FOCUS_PROD", "FOCUS_EVENT", "FOCUS_SPACE",
        "FOCUS_PEOPLE", "FOCUS_OFFER", "FOCUS_ROUTINE"
    ],
    "Нарративы пространства": [
        "NARR_PLACE_THIRD", "NARR_PLACE_EXPERIENCE", "NARR_PLACE_UTILITY",
        "NARR_PLACE_EVENTFUL", "NARR_PLACE_COMMUNITY", "NARR_PLACE_LOCAL",
        "NARR_PLACE_ESCAPE", "NARR_PLACE_CONTRAST", "NARR_PLACE_EXOTIC",
        "NARR_PLACE_HOME"
    ],
    "Механика": [
        "MECH_CALL_DIRECT", "MECH_CALL_SOFT", "MECH_BENEFIT",
        "MECH_EMOTION", "MECH_SCARCITY", "MECH_NOVELTY", "MECH_RITUAL",
        "MECH_DIALOGUE", "MECH_SOCIAL_PROOF", "MECH_TRADITION",
        "MECH_SENSORY_HOOK"
    ],
    "Тон": [
        "TONE_PUSH_HARD", "TONE_PUSH_SOFT",
        "TONE_PULL_AESTH", "TONE_PULL_BELONG", "TONE_PULL_VALUE"
    ],
}


## 5. Описательная статистика кодов

Доля постов с присутствием каждого кода в периоде ПН и вне ПН, разница в процентных пунктах.

In [ ]:
ctrl = df[df["period_bin"] == 0]
exp  = df[df["period_bin"] == 1]

for block_name, cols in ALL_BLOCKS.items():
    existing = [c for c in cols if c in df.columns]
    freq = pd.DataFrame({
        "вне ПН, %": ctrl[existing].mean() * 100,
        "ПН, %":     exp[existing].mean()  * 100,
    })
    freq["delta_pp"] = (freq["ПН, %"] - freq["вне ПН, %"]).round(2)
    freq = freq.sort_values("delta_pp", ascending=False).round(2)
    print(f"\n=== {block_name} ===")
    display(freq)


## 6. Проверка мультиколлинеарности

Попарные корреляции Спирмена между предикторами внутри каждого блока. Порог допустимости: |r| < 0.5.

In [ ]:
for block_name, cols in ALL_BLOCKS.items():
    existing = [c for c in cols if c in df.columns]
    if len(existing) < 2:
        continue
    corr_mat = df[existing].rank().corr(method="spearman")
    upper = corr_mat.where(
        np.triu(np.ones(corr_mat.shape), k=1).astype(bool)
    )
    max_r = upper.abs().max().max()
    print(f"{block_name}: макс. |r| = {max_r:.3f}")


## 7. Функция бинарной логистической регрессии

Принимает датафрейм и список предикторов одного блока; возвращает таблицу OR с 95-процентными ДИ и метрики качества модели.

In [ ]:
def run_logit(df, predictors, dep="period_bin"):
    existing = [c for c in predictors if c in df.columns]
    sub = df[existing + [dep]].dropna()
    X = sm.add_constant(sub[existing].astype(float))
    y = sub[dep].astype(int)
    model = Logit(y, X).fit(disp=False)

    or_df = pd.DataFrame({
        "predictor": existing,
        "beta":      model.params[existing].values,
        "OR":        np.exp(model.params[existing]).values,
        "LCL_OR":    np.exp(model.conf_int().loc[existing, 0]).values,
        "UCL_OR":    np.exp(model.conf_int().loc[existing, 1]).values,
        "p":         model.pvalues[existing].values,
    }).sort_values("beta", ascending=False)

    def sig_label(p):
        if p < 0.001: return "***"
        if p < 0.01:  return "**"
        if p < 0.05:  return "*"
        if p < 0.1:   return "~"
        return "n.s."
    or_df["sig"] = or_df["p"].apply(sig_label)

    proba   = model.predict(X)
    auc     = roc_auc_score(y, proba)
    quality = {"McFadden_R2": round(model.prsquared, 4),
               "AUC_ROC":     round(auc, 4),
               "N":           len(sub)}
    return or_df.round(4), quality


## 8. Результаты регрессии по блокам

### 8.1. Блок «Ритмика и сенсорика»

In [ ]:
or_df, m = run_logit(df, ['SYNC', 'TIME_WEEKEND', 'TIME_EVENING', 'CONTEXT_WEATHER', 'CONTEXT_LIGHT', 'CONTEXT_SEASON', 'SENSORY_BODY', 'SENSORY_WARM', 'SENSORY_LIGHT'])
print("McFadden R2:", m["McFadden_R2"], "  AUC-ROC:", m["AUC_ROC"], "  N:", m["N"])
display(or_df)


### 8.2. Блок «Функции»

In [ ]:
or_df, m = run_logit(df, ['FUNC_PRM', 'FUNC_ENG', 'FUNC_IMG', 'FUNC_COM', 'FUNC_INF'])
print("McFadden R2:", m["McFadden_R2"], "  AUC-ROC:", m["AUC_ROC"], "  N:", m["N"])
display(or_df)


### 8.3. Блок «Фокус»

In [ ]:
or_df, m = run_logit(df, ['FOCUS_PROD', 'FOCUS_EVENT', 'FOCUS_SPACE', 'FOCUS_PEOPLE', 'FOCUS_OFFER', 'FOCUS_ROUTINE'])
print("McFadden R2:", m["McFadden_R2"], "  AUC-ROC:", m["AUC_ROC"], "  N:", m["N"])
display(or_df)


### 8.4. Блок «Нарративы пространства»

In [ ]:
or_df, m = run_logit(df, ['NARR_PLACE_THIRD', 'NARR_PLACE_EXPERIENCE', 'NARR_PLACE_UTILITY', 'NARR_PLACE_EVENTFUL', 'NARR_PLACE_COMMUNITY', 'NARR_PLACE_LOCAL', 'NARR_PLACE_ESCAPE', 'NARR_PLACE_CONTRAST', 'NARR_PLACE_EXOTIC', 'NARR_PLACE_HOME'])
print("McFadden R2:", m["McFadden_R2"], "  AUC-ROC:", m["AUC_ROC"], "  N:", m["N"])
display(or_df)


### 8.5. Блок «Механика»

In [ ]:
or_df, m = run_logit(df, ['MECH_CALL_DIRECT', 'MECH_CALL_SOFT', 'MECH_BENEFIT', 'MECH_EMOTION', 'MECH_SCARCITY', 'MECH_NOVELTY', 'MECH_RITUAL', 'MECH_DIALOGUE', 'MECH_SOCIAL_PROOF', 'MECH_TRADITION', 'MECH_SENSORY_HOOK'])
print("McFadden R2:", m["McFadden_R2"], "  AUC-ROC:", m["AUC_ROC"], "  N:", m["N"])
display(or_df)


### 8.6. Блок «Тон»

In [ ]:
or_df, m = run_logit(df, ['TONE_PUSH_HARD', 'TONE_PUSH_SOFT', 'TONE_PULL_AESTH', 'TONE_PULL_BELONG', 'TONE_PULL_VALUE'])
print("McFadden R2:", m["McFadden_R2"], "  AUC-ROC:", m["AUC_ROC"], "  N:", m["N"])
display(or_df)


## 9. Сводная таблица метрик качества моделей

In [ ]:
summary_rows = []
_, _m = run_logit(df, ['SYNC', 'TIME_WEEKEND', 'TIME_EVENING', 'CONTEXT_WEATHER', 'CONTEXT_LIGHT', 'CONTEXT_SEASON', 'SENSORY_BODY', 'SENSORY_WARM', 'SENSORY_LIGHT'])
summary_rows.append({"Блок": 'Ритмика и сенсорика', "McFadden R2": _m["McFadden_R2"], "AUC-ROC": _m["AUC_ROC"], "N": _m["N"]})
_, _m = run_logit(df, ['FUNC_PRM', 'FUNC_ENG', 'FUNC_IMG', 'FUNC_COM', 'FUNC_INF'])
summary_rows.append({"Блок": 'Функции', "McFadden R2": _m["McFadden_R2"], "AUC-ROC": _m["AUC_ROC"], "N": _m["N"]})
_, _m = run_logit(df, ['FOCUS_PROD', 'FOCUS_EVENT', 'FOCUS_SPACE', 'FOCUS_PEOPLE', 'FOCUS_OFFER', 'FOCUS_ROUTINE'])
summary_rows.append({"Блок": 'Фокус', "McFadden R2": _m["McFadden_R2"], "AUC-ROC": _m["AUC_ROC"], "N": _m["N"]})
_, _m = run_logit(df, ['NARR_PLACE_THIRD', 'NARR_PLACE_EXPERIENCE', 'NARR_PLACE_UTILITY', 'NARR_PLACE_EVENTFUL', 'NARR_PLACE_COMMUNITY', 'NARR_PLACE_LOCAL', 'NARR_PLACE_ESCAPE', 'NARR_PLACE_CONTRAST', 'NARR_PLACE_EXOTIC', 'NARR_PLACE_HOME'])
summary_rows.append({"Блок": 'Нарративы пространства', "McFadden R2": _m["McFadden_R2"], "AUC-ROC": _m["AUC_ROC"], "N": _m["N"]})
_, _m = run_logit(df, ['MECH_CALL_DIRECT', 'MECH_CALL_SOFT', 'MECH_BENEFIT', 'MECH_EMOTION', 'MECH_SCARCITY', 'MECH_NOVELTY', 'MECH_RITUAL', 'MECH_DIALOGUE', 'MECH_SOCIAL_PROOF', 'MECH_TRADITION', 'MECH_SENSORY_HOOK'])
summary_rows.append({"Блок": 'Механика', "McFadden R2": _m["McFadden_R2"], "AUC-ROC": _m["AUC_ROC"], "N": _m["N"]})
_, _m = run_logit(df, ['TONE_PUSH_HARD', 'TONE_PUSH_SOFT', 'TONE_PULL_AESTH', 'TONE_PULL_BELONG', 'TONE_PULL_VALUE'])
summary_rows.append({"Блок": 'Тон', "McFadden R2": _m["McFadden_R2"], "AUC-ROC": _m["AUC_ROC"], "N": _m["N"]})
display(pd.DataFrame(summary_rows))


## 10. Корреляционные матрицы Спирмена внутри блоков

In [ ]:
for block_name, cols in ALL_BLOCKS.items():
    existing = [c for c in cols if c in df.columns]
    if len(existing) < 2:
        continue
    corr_mat = df[existing].rank().corr(method="spearman").round(3)
    print(f"\n=== {block_name} ===")
    display(corr_mat)
